# Prerequisite - Score Based Diffusion Model

Diffusion model is an effective tool to sample from a distribution $p(x)$, by starting from sampling from a template noise distribution, e.g. $\mathcal{N}(0, I)$, then denoise the noisy sample with a denoiser. To describe it more precisely, the template noise distribution is in fact a result of data distribution of a forward process of a stochastic differential equation:

$$
dx_t = f(t)x_t\,dt + g(t)\,dw_t
$$

where $x_0 \sim p_{\mathrm{data}}(x)$, scalar functions $f(t), g(t) : \mathbb{R} \to \mathbb{R}$ denote the **drift** and **diffusion coefficients**, respectively, and $\{w_t\}_{t \in [0,1]}$ is the standard Wiener process.

Thus, the sampling process is in fact a reverse process of the above SDE, using the following reverse-time ODE.

$$
dx_t = \left( f(t)x_t - \frac{1}{2} g^2(t)\nabla \log p_t(x_t) \right)\,dt.
$$

Unfortunately, the score function $\nabla_{x_t} \log p_t(x_t)$ at each $t \in [0,1]$ is typically unknown. We therefore estimate the posterior mean $\mathbb{E}[x_0 \mid x_t]$ as an alternative approach for estimating the score function, in which Tweedie's formula establishes an equivalence between the score function and posterior mean as follows:

$$
\mathbb{E}[\boldsymbol{x} \mid \boldsymbol{x}_t] = \frac{1}{\alpha_t}\left(\boldsymbol{x}_t + \sigma_t^2 \nabla \log p_t(\boldsymbol{x}_t)\right)
$$


# Conditional inference if we get ahold of the prior $p(x)$

Almost all application in machine learning can be viewed as a special case of the following inference problem. Given:
$$y = h(x) + w$$
where $h(\cdot)$ represents measurement of $x$ and $w$ represents some measurement noise and even sparse corruptions, solve the inverse problem of obtaining the most likely $x$ or sample $\hat{x}$ that is at least consistent with the observation. 



In this notebook, we care about the scenario which we only are provided paired data $(x,y)$.

By Bayes' rule and conditional independence of $y$ and $x_t$ given $x$, the optimal conditional denoiser can be written as

$$
\bar{\boldsymbol{x}}_\theta^{\text{CG, ideal}}(t, \boldsymbol{\xi}, \nu)
  = \mathbb{E}[\boldsymbol{x} \mid \boldsymbol{x}_t = \boldsymbol{\xi}]
  + \gamma \frac{\sigma_t^2}{\alpha_t}\,\nabla_{\boldsymbol{\xi}} \log p_{y \mid \boldsymbol{x}_t}(\nu \mid \boldsymbol{\xi})
$$


# Conditional inference with classifier free guidance

If we assume the classifier perfectly approximate the targets, we obtain the classifer free guidance denoiser:

$$
\bar{\boldsymbol{x}}_\theta^{\text{CFG}}(t, \boldsymbol{x}_t, y)
  = (1-\gamma)\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, \varnothing)
  + \gamma\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, y)
$$

where $\gamma > 1$ which there is solid explannation of why large $\gamma > 1$ is essential but I am not documenting it here. 

## CFG denoiser for GMM

We will use the same network to represent both conditional and unconditional denoiser.

Given any distribution for $x$, there exists a Gaussian mixture that approximates it arbitrarily well. Here, let's consider the parameterization of denoiser for GMM.

$$
x \sim \frac{1}{K}\sum_{k=1}^{K} \mathcal{N}(0, U_k U_k^{\top})
$$

Recall:

$$
\bar{\boldsymbol{x}}_\theta^{\text{CFG}}(t, \boldsymbol{x}_t, y)
  = (1-\gamma)\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, \varnothing)
  + \gamma\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, y)
$$

For GMM case, 

$$\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, \varnothing) = \frac{1}{1+t^2}
\sum_{k=1}^{K}
\frac{
  \exp\!\left(
    \frac{1}{2t^2(1+t^2)}\,
    \bigl\|U_k^\top \boldsymbol{x}_t\bigr\|_2^2
  \right)
}{
  \displaystyle
  \sum_{i=1}^{K}
  \exp\!\left(
    \frac{1}{2t^2(1+t^2)}\,
    \bigl\|U_i^\top \boldsymbol{x}_t\bigr\|_2^2
  \right)
}
\,U_k U_k^\top \boldsymbol{x}_t$$



$$\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, y) = \frac{1}{1+t^2} U_y U_y^\top \boldsymbol{x}_t$$

$$\bar{\boldsymbol{x}}^{\text{CFG, ideal}}(t, \boldsymbol{x}_t, y) = \frac{1}{1+t^2}\left( (1-\gamma)\sum_{k=1}^{K} \frac{\exp\!\left(\frac{1}{2t^2(1+t^2)}\|\boldsymbol{U}_k^\top \boldsymbol{x}_t\|_2^2\right)}{\sum_{i=1}^{K} \exp\!\left(\frac{1}{2t^2(1+t^2)}\|\boldsymbol{U}_i^\top \boldsymbol{x}_t\|_2^2\right)}\,\boldsymbol{U}_k\boldsymbol{U}_k^\top + \gamma\,\boldsymbol{U}_y\boldsymbol{U}_y^\top \right)\boldsymbol{x}_t$$

We are now going to come up with one single operator that can be used to represent both $\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, \varnothing)$ and $\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, y) $

Let's make some assumptions

Now we consider the problem of parameterizing a learnable denoiser $\boldsymbol{x}_\theta^{\text{CFG}}$ to represent the optimal denoiser. For tractability, we add an additional assumption associated to the subspaces $\boldsymbol{U}_k$ being ‘distinguishable’ from one another, which is natural in practice: specifically, we assume that for any pair of indices $k, k' \in [K]$ with $k \neq k'$, we can find a set of $K$ nonzero directions $\boldsymbol{v}_k \in \mathbb{R}^D$ such that
$$
\boldsymbol{U}_k \boldsymbol{U}_k^\top \boldsymbol{v}_k = \boldsymbol{v}_k,\quad
\boldsymbol{U}_{k'} \boldsymbol{U}_{k'}^\top \boldsymbol{v}_k = \boldsymbol{0},\quad
k' \neq k.
$$

These vectors $\boldsymbol{v}_k$ can be considered as embeddings of class label $y$. It also comes with a benefit to map $y$ into a space of same dimension as where $x_t$ lives in. We can then come up with a general operator to represent both $\,\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, \varnothing)$ and $\bar{\boldsymbol{x}}_\theta(t, \boldsymbol{x}_t, y) $:

$$
(x_t, \boldsymbol{v}) \mapsto
\sum_{k=1}^{K}
\frac{
  \exp\!\left(
    \dfrac{1}{2t^2(1+t^2)}\,
    \bigl|x_t^\top U_k U_k^\top \boldsymbol{v}\bigr|
  \right)
}{
  \displaystyle\sum_{i=1}^{K}
  \exp\!\left(
    \dfrac{1}{2t^2(1+t^2)}\,
    \bigl|x_t^\top U_i U_i^\top \boldsymbol{v}\bigr|
  \right)
}
\,U_k U_k^\top x_t
$$

If you plug in $v = x_t$, you will get exactly the unconditional denoiser and plugging in $v = v_y$ you will get class-conditional denoiser for $y$